## SRP166732

**paper:** [no paper]() - 

**date, curator:** 2026-04-10, Sara Carsanaro

**notes**
* removed lung and kidney libraries - the animal died of leptospirosis and streptococcal pneumonia

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,small intestine,UBERON:0002108,small intestine,perfect match
1,spleen,UBERON:0002106,spleen,perfect match
2,blubber,UBERON:0009754,blubber,perfect match
3,heart,UBERON:0000948,heart,perfect match
4,adrenal gland (both the cortex and medulla),UBERON:0002369,adrenal gland,perfect match
5,hippocampus,UBERON:0002421,hippocampal formation,perfect match
6,testis,UBERON:0000473,testis,perfect match
7,liver,UBERON:0002107,liver,perfect match
8,"six brain regions: thalamus, striatum, hypothalamus, amygdala, cerebellum, cerebellar nuclei",UBERON:0000955,brain,other


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,juvenile,UBERON:0034919,juvenile stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP166732"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 11/11 [00:12<00:00,  1.09s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,,,,small intestine,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 small intestine,SAMN10285337,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 small intestine,,,,juvenile,,TRANSCRIPTOMIC,unspecified
1,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,,,,spleen,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 spleen,SAMN10285336,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 spleen,,,,juvenile,,TRANSCRIPTOMIC,unspecified
2,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,,,,blubber,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 blubber,SAMN10285333,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 blubber,,,,juvenile,,TRANSCRIPTOMIC,unspecified
3,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,,,,heart,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 heart,SAMN10285332,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 heart,,,,juvenile,,TRANSCRIPTOMIC,unspecified
4,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,,,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 adrenal gland,,,,juvenile,,TRANSCRIPTOMIC,unspecified
5,SRX4928798,SRP166732,Illumina HiSeq 4000,SRS3973806,UBERON:0002421,hippocampal formation,,,,hippocampus,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 hippocampus,SAMN10285334,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 hippocampus,,,,juvenile,,TRANSCRIPTOMIC,unspecified
6,SRX4928796,SRP166732,Illumina HiSeq 4000,SRS3973804,UBERON:0000473,testis,,,,testis,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 testis,SAMN10285328,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 testis,,,,juvenile,,TRANSCRIPTOMIC,unspecified
7,SRX4928794,SRP166732,Illumina HiSeq 4000,SRS3973803,UBERON:0002107,liver,,,,liver,juvenile,perfect match,not documented,,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 liv

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['adrenal gland (both the cortex and medulla)' 'blubber' 'heart'
 'hippocampus' 'liver'
 'six brain regions: thalamus, striatum, hypothalamus, amygdala, cerebellum, cerebellar nuclei'
 'small intestine' 'spleen' 'testis']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['juvenile']


In [7]:
# all
library.loc[:,'stageId'] = 'UBERON:0034919'
library.loc[:,'stageName'] = 'juvenile stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,UBERON:0034919,juvenile stage,,small intestine,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 small intestine,SAMN10285337,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 small intestine,,,,juvenile,,TRANSCRIPTOMIC,unspecified
1,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,spleen,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 spleen,SAMN10285336,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 spleen,,,,juvenile,,TRANSCRIPTOMIC,unspecified
2,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,UBERON:0034919,juvenile stage,,blubber,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 blubber,SAMN10285333,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 blubber,,,,juvenile,,TRANSCRIPTOMIC,unspecified
3,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,UBERON:0034919,juvenile stage,,heart,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 heart,SAMN10285332,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 heart,,,,juvenile,,TRANSCRIPTOMIC,unspecified
4,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,UBERON:0034919,juvenile stage,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 adrenal gland,,,,juvenile,,TRANSCRIPTOMIC,unspecified
5,SRX4928798,SRP166732,Illumina HiSeq 4000,SRS3973806,UBERON:0002421,hippocampal formation,UBERON:0034919,juvenile stage,,hippocampus,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 hippocampus,SAMN10285334,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 hippocampus,,,,juvenile,,TRANSCRIPTOMIC,unspecified
6,SRX4928796,SRP166732,Illumina HiSeq 4000,SRS3973804,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,testis,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,CSL-13825 testis,SAMN10285328,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RN

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,UBERON:0034919,juvenile stage,,small intestine,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 small intestine,SAMN10285337,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 small intestine,,,,juvenile,,TRANSCRIPTOMIC,unspecified
1,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,spleen,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 spleen,SAMN10285336,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 spleen,,,,juvenile,,TRANSCRIPTOMIC,unspecified
2,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,UBERON:0034919,juvenile stage,,blubber,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 blubber,SAMN10285333,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 blubber,,,,juvenile,,TRANSCRIPTOMIC,unspecified
3,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,UBERON:0034919,juvenile stage,,heart,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 heart,SAMN10285332,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 heart,,,,juvenile,,TRANSCRIPTOMIC,unspecified
4,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,UBERON:0034919,juvenile stage,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 adrenal gland,,,,juvenile,,TRANSCRIPTOMIC,unspecified
5,SRX4928798,SRP166732,Illumina HiSeq 4000,SRS3973806,UBERON:0002421,hippocampal formation,UBERON:0034919,juvenile stage,,hippocampus,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 hippocampus,SAMN10285334,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 hippocampus,,,,juvenile,,TRANSCRIPTOMIC,unspecified
6,SRX4928796,SRP166732,Illumina HiSeq 4000,SRS3973804,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,testis,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 testis,SAMN10285328,,,,,,,,leptospirosis and streptococcal pneumonia,,,10/04/2026,Directional libraries were constructed using the NEBNext Ultra 

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,UBERON:0034919,juvenile stage,,small intestine,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 small intestine,SAMN10285337,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 small intestine,,,,juvenile,,TRANSCRIPTOMIC,unspecified
1,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,spleen,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 spleen,SAMN10285336,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 spleen,,,,juvenile,,TRANSCRIPTOMIC,unspecified
2,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,UBERON:0034919,juvenile stage,,blubber,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 blubber,SAMN10285333,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 blubber,,,,juvenile,,TRANSCRIPTOMIC,unspecified
3,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,UBERON:0034919,juvenile stage,,heart,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 heart,SAMN10285332,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 heart,,,,juvenile,,TRANSCRIPTOMIC,unspecified
4,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,UBERON:0034919,juvenile stage,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 adrenal gland,,,,juvenile,,TRANSCRIPTOMIC,unspecified
5,SRX4928798,SRP166732,Illumina HiSeq 4000,SRS3973806,UBERON:0002421,hippocampal formation,UBERON:0034919,juvenile stage,,hippocampus,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 hippocampus,SAMN10285334,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed using the NEBNext Ultra Directional RNA Library Prep Kit,,CSL-13825 hippocampus,,,,juvenile,,TRANSCRIPTOMIC,unspecified
6,SRX4928796,SRP166732,Illumina HiSeq 4000,SRS3973804,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,testis,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 testis,SAMN10285328,,,,,,,,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10,Directional libraries were constructed usi

#### comments

In [11]:
library.loc[:,'comment'] = 'no paper'

#### save complete file with correct columns

In [12]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,UBERON:0034919,juvenile stage,,small intestine,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 small intestine,SAMN10285337,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
1,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,spleen,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 spleen,SAMN10285336,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
2,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,UBERON:0034919,juvenile stage,,blubber,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 blubber,SAMN10285333,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
3,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,UBERON:0034919,juvenile stage,,heart,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 heart,SAMN10285332,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
4,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,UBERON:0034919,juvenile stage,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
5,SRX4928798,SRP166732,Illumina HiSeq 4000,SRS3973806,UBERON:0002421,hippocampal formation,UBERON:0034919,juvenile stage,,hippocampus,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 hippocampus,SAMN10285334,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
6,SRX4928796,SRP166732,Illumina HiSeq 4000,SRS3973804,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,testis,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 testis,SAMN10285328,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
7,SRX4928794,SRP166732,Illumina HiSeq 4000,SRS3973803,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 liver,SAMN10285330,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
8,SRX4928793,SRP166732,Illumina HiSeq 4000,SRS3973801,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,"six brain regions: thalamus, striatum, hypothalamus, amygdala, cerebellum, cerebellar nuclei",juvenile,other,partial sampling,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 brain - not hippocampus,SAMN10285338,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10


### experiment annotations

In [13]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP166732,Strand-specific transcriptome sequencing of multiple tissues from California sea lion (Zalophus californianus),"Strand-specific transcriptomic data was acquired from different tissues from a California sea lion (Zalophus californianus) to provide evidence for gene prediction when annotating genome assemblies. Tissues were collected post-mortem at The Marine Mammal Center (Sausalito, CA) under Marine Mammal Protection Act (MMPA) permit number 18786. The animal (Ensign; CSL-13825) was a juvenile male who was euthanized because of poor response to treatment for leptospirosis and streptococcal pneumonia. The sequencing project was reviewed and approved by the National Institute of Standards and Technology Animal Care and Use Coordinator (ACUC), project MML-AR18-0002.",SRA,,,,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [14]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

9

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP166732,Strand-specific transcriptome sequencing of multiple tissues from California sea lion (Zalophus californianus),"Strand-specific transcriptomic data was acquired from different tissues from a California sea lion (Zalophus californianus) to provide evidence for gene prediction when annotating genome assemblies. Tissues were collected post-mortem at The Marine Mammal Center (Sausalito, CA) under Marine Mammal Protection Act (MMPA) permit number 18786. The animal (Ensign; CSL-13825) was a juvenile male who was euthanized because of poor response to treatment for leptospirosis and streptococcal pneumonia. The sequencing project was reviewed and approved by the National Institute of Standards and Technology Animal Care and Use Coordinator (ACUC), project MML-AR18-0002.",SRA,partial,Bgee 1K,9,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
#experiment.loc[:,'reference_url'] = ''
experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP166732,Strand-specific transcriptome sequencing of multiple tissues from California sea lion (Zalophus californianus),"Strand-specific transcriptomic data was acquired from different tissues from a California sea lion (Zalophus californianus) to provide evidence for gene prediction when annotating genome assemblies. Tissues were collected post-mortem at The Marine Mammal Center (Sausalito, CA) under Marine Mammal Protection Act (MMPA) permit number 18786. The animal (Ensign; CSL-13825) was a juvenile male who was euthanized because of poor response to treatment for leptospirosis and streptococcal pneumonia. The sequencing project was reviewed and approved by the National Institute of Standards and Technology Animal Care and Use Coordinator (ACUC), project MML-AR18-0002.",SRA,partial,Bgee 1K,9,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,,,


#### comments

In [17]:
experiment.loc[:,'comment'] = 'removed two libraries due to infection, the corresponding tissues were rejected'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP166732,Strand-specific transcriptome sequencing of multiple tissues from California sea lion (Zalophus californianus),"Strand-specific transcriptomic data was acquired from different tissues from a California sea lion (Zalophus californianus) to provide evidence for gene prediction when annotating genome assemblies. Tissues were collected post-mortem at The Marine Mammal Center (Sausalito, CA) under Marine Mammal Protection Act (MMPA) permit number 18786. The animal (Ensign; CSL-13825) was a juvenile male who was euthanized because of poor response to treatment for leptospirosis and streptococcal pneumonia. The sequencing project was reviewed and approved by the National Institute of Standards and Technology Animal Care and Use Coordinator (ACUC), project MML-AR18-0002.",SRA,partial,Bgee 1K,9,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,,,"removed two libraries due to infection, the corresponding tissues were rejected"


#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
60960,SRX9803743,SRP300817,Illumina HiSeq 2000,SRS7987969,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,9689,,,polyA,,1,Ple_8,SAMN17252893,,,,,,,no paper,,,SAC,2026-04-10
60961,SRX9803742,SRP300817,Illumina HiSeq 2000,SRS7987969,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,9689,,,polyA,,1,Ple_8,SAMN17252893,,,,,,,no paper,,,SAC,2026-04-10
60962,SRX4928803,SRP166732,Illumina HiSeq 4000,SRS3973811,UBERON:0002108,small intestine,UBERON:0034919,juvenile stage,,small intestine,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 small intestine,SAMN10285337,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
60963,SRX4928802,SRP166732,Illumina HiSeq 4000,SRS3973810,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,spleen,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 spleen,SAMN10285336,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
60964,SRX4928801,SRP166732,Illumina HiSeq 4000,SRS3973809,UBERON:0009754,blubber,UBERON:0034919,juvenile stage,,blubber,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 blubber,SAMN10285333,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
60965,SRX4928800,SRP166732,Illumina HiSeq 4000,SRS3973808,UBERON:0000948,heart,UBERON:0034919,juvenile stage,,heart,juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 heart,SAMN10285332,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10
60966,SRX4928799,SRP166732,Illumina HiSeq 4000,SRS3973807,UBERON:0002369,adrenal gland,UBERON:0034919,juvenile stage,,adrenal gland (both the cortex and medulla),juvenile,perfect match,not documented,perfect match,M,,,9704,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,CSL-13825 adrenal gland,SAMN10285335,,,,,,,no paper,leptospirosis and streptococcal pneumonia,,SAC,2026-04-10


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1163,SRP105391,Panthera leo persica Transcriptome sequencing,First transcriptome sequencing of Asiatic lion...,SRA,total,Bgee 1K,5,,,,PRJNA384102,,https://www.biorxiv.org/content/10.1101/549790...,10.1101/549790,,no PMID it appears
1164,SRP300817,RNA-Seq from lion whole testis,Whole testis RNA-seq data generated from indiv...,SRA,total,Bgee 1K,2,,,,PRJNA690586,,,,,no paper
1165,SRP166732,Strand-specific transcriptome sequencing of mu...,Strand-specific transcriptomic data was acquir...,SRA,partial,Bgee 1K,9,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,,,"removed two libraries due to infection, the co..."


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 5a7b61c] adding annotated bulk experiment SRP166732
 2 files changed, 10 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.68 KiB | 2.68 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   c4a39be..5a7b61c  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push